# Uncertainty Quantification for LLMs: A Comprehensive Tutorial

Welcome to this interactive tutorial on estimating uncertainty in Large Language Models (LLMs) applied to the clinical domain.

This notebook provides a hands-on approach to Uncertainty Quantification (UQ) by leveraging two state-of-the-art Python libraries: **`lm-polygraph`** and **`uqlm`**.

##### Load  the requested libraries

In [4]:
# CELL 0: ENVIRONMENT SETUP (Run this first!)
!pip install -q lm-polygraph uqlm transformers accelerate langchain langchain-huggingface langchain_openai

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 60.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 10.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.9/92.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 17.5 MB/s eta 0:0

In [1]:
import lm_polygraph
import uqlm
import torch
import random
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import getpass


##### Fix a random seed to ensure reproducibility of your experiment

In [2]:
# --- Setting seed for Scientific Reproducibility ---
RANDOM_SEED = 42

def set_global_seed(seed=RANDOM_SEED):
    """Locks the random seed for predictable, reproducible results."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_global_seed()


## Model selection
### Setup the Control Panel
In this first phase, we lay the foundations for our *Uncertainty Quantification* experiment. To ensure maximum reproducibility and clean code, we have created a **centralized Control Panel**. By modifying the four main variables, you can reconfigure the entire architecture without touching the underlying logic.

**1. Imports (The Wrappers)**
* We import the `BlackboxModel` and `WhiteboxModel` classes for the `lm_polygraph` library.
* We import the *Scorers* for the `uqlm` library, alongside the **LangChain** adapters (`ChatOpenAI`, `ChatHuggingFace`). LangChain acts as a universal bridge, allowing `uqlm` to communicate with any external API using a standard format.

**2. Configuration Variables**
* `PROVIDER`: The main switch. Choose `"openai"` or `"huggingface"` to route all requests to the respective servers.
  > 💡 **Why only these two providers?** While `uqlm` (thanks to LangChain) supports dozens of API providers, `lm_polygraph` currently only supports OpenAI and Hugging Face for Black/Grey-box inference. We deliberately restricted the scope to these two providers to ensure a rigorously fair, 1:1 comparative analysis between the two libraries.
* `MODEL_ID`: The exact name of the model you want to test (e.g., `"google/gemma-2-2b-it"` or `"gpt-3.5-turbo"`).
* `POLYGRAPH_MODE`: Choose `"white"` to download the model's weights into local memory (VRAM), unlocking attention matrices and hidden states, or `"black"` to query the model via API.
* `UQLM_MODE`: Choose `"white"` to force the APIs to return token probabilities (*Logprobs*), or `"black"` to evaluate uncertainty purely by analyzing the generated text.

> ⚠️ **Crucial Note on UQ Availability and Access Levels:** > The mode you select directly dictates the arsenal of Uncertainty Quantification techniques at your disposal. This is a strict hierarchy:
> * **White-Box Mode (Total Access):** By downloading the weights locally, you gain complete mathematical access to the model's internals. Because you have the highest level of access, a White-box model can execute **ALL** UQ methods (White-box  and Black-box).
> * **Black-Box Mode (Restricted Access):** This restricts access purely to the external API outputs. Consequently, you are limited **ONLY** to Black-box statistical methods (like Self-Consistency or text-based semantic similarity). You cannot apply attention-based or entropy-based methods here unless the API explicitly supports probability extraction (Grey-box).

In [4]:
# --- Imports for Wrapper A (lm_polygraph) ---
from lm_polygraph.utils.model import BlackboxModel, WhiteboxModel
# --- Imports for Wrapper B (uqlm via LangChain) ---
from uqlm.scorers.black_box import BlackBoxUQ
from uqlm.scorers.white_box import WhiteBoxUQ
from uqlm.scorers.longtext import LongTextUQ

from langchain_openai import ChatOpenAI
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

PROVIDER = "huggingface"          # Options: "openai" or "huggingface"
MODEL_ID = "google/gemma-2-2b-it" # Target model (e.g., "gpt-3.5-turbo" or "google/gemma-2-2b-it")


In [4]:
while True:
    POLYGRAPH_MODE = input("Select POLYGRAPH_MODE ('white' or 'black'): ").strip().lower()
    if POLYGRAPH_MODE in ['white', 'black']:
        break
    print("Invalid input. Please type exactly 'white' or 'black'.")

# 2. Ask for UQLM Mode with validation
while True:
    UQLM_MODE = input("Select UQLM_MODE ('white' or 'black'): ").strip().lower()
    if UQLM_MODE in ['white', 'black']:
        break
    print("Invalid input. Please type exactly 'white' or 'black'.")

Select POLYGRAPH_MODE ('white' or 'black'): white
Select UQLM_MODE ('white' or 'black'): white


### Sanity Checks & Secure Authentication
Before allocating heavy resources or initiating network requests, this block ensures that the chosen configuration is logically sound and securely authenticated. This proactive approach prevents unexpected runtime crashes.

**1. Architectural Sanity Checks**
Not all configurations are physically possible. For instance, OpenAI models (like GPT-4 or GPT-3.5) are proprietary and closed-source. It is impossible to download their weights into your local VRAM. If a user accidentally sets `PROVIDER = "openai"` alongside `POLYGRAPH_MODE = "white"`, the script will immediately catch the logical conflict and raise a clear `ValueError`, guiding the user to correct the setup.

**2. Dynamic and Secure Credential Injection**
Security and efficiency are paramount. Instead of hardcoding sensitive API keys or asking for tokens you don't need:
* The script evaluates your `PROVIDER` and `MODE` choices.
* It dynamically prompts you **only** for the specific credentials required by your active configuration (e.g., it won't ask for a Hugging Face token if you are only querying the OpenAI API).
* It uses the `getpass` module to securely mask your input, ensuring that your private keys are never exposed in the notebook's output history or saved state.

In [5]:
print(f"Initializing Master Architecture for {MODEL_ID}...")

# Prevent impossible architectural states
if PROVIDER == "openai" and POLYGRAPH_MODE == "white":
    raise ValueError("Conflict: OpenAI models cannot be downloaded to local VRAM. Set POLYGRAPH_MODE='black'.")

# Request exactly the keys needed for the chosen configuration
if PROVIDER == "openai" and "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API Key: ")

if (PROVIDER == "huggingface" or POLYGRAPH_MODE == "white") and "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass.getpass("Paste your Hugging Face Token: ")

Initializing Master Architecture for google/gemma-2-2b-it...
Paste your Hugging Face Token: ··········


###Building the UQ Wrappers (Polygraph & UQLM)
Now we bring the architecture to life by constructing the specific **Wrappers** required by each library. These wrappers act as the crucial translation layer, converting raw LLM outputs into a format that the Uncertainty Quantification (UQ) algorithms can mathematically process.

**1. Wrapper A: The `lm_polygraph` Interfaces**
* **`WhiteboxModel`:** This wrapper directly ingests the raw, locally downloaded model weights (`base_model`) and its `tokenizer`. By wrapping the physical model, it exposes the deepest mathematical layers of the LLM—such as hidden states, attention matrices, and logits—directly to Polygraph's advanced estimators.
* **`BlackboxModel`:** Instead of local weights, this wrapper encapsulates an external API connection. While it typically treats the LLM as a pure text-in/text-out oracle, setting `supports_logprobs=True` gracefully upgrades it into a "Grey-box." This allows Polygraph to compute token-level entropy using API probabilities without ever needing the physical model.

**2. Wrapper B: The `uqlm` Interfaces (via LangChain)**
* **The LangChain Bridge:** Unlike Polygraph, `uqlm` does not interface with models directly. It requires a LangChain adapter (`ChatOpenAI` or `ChatHuggingFace`) to wrap the model, standardizing the connection regardless of the underlying provider.
* **`WhiteBoxUQ` Scorer:** In the `uqlm` dictionary, "White-box" means probability-based. This wrapper takes the LangChain object and leverages extracted `logprobs` from the API, enabling fast, single-generation mathematical scoring.
* **`BlackBoxUQ` Scorer:** This wrapper ignores probabilities entirely. It wraps the model to perform purely text-based sampling UQ, evaluating uncertainty based on the semantic consistency across multiple generated responses.

In [6]:
from lm_polygraph.utils.generation_parameters import GenerationParameters

print(f" Setting up lm_polygraph in {POLYGRAPH_MODE.upper()}-BOX mode...")
shared_generation_params = GenerationParameters()
shared_generation_params.temperature = 0.7
shared_generation_params.do_sample = True
shared_generation_params.max_new_tokens = 256

if POLYGRAPH_MODE == "white":
    # Load heavy weights into GPU exactly once (16-bit precision)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ["HF_TOKEN"])
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="cuda:0",
        torch_dtype=torch.float16,
        token=os.environ["HF_TOKEN"]
    )
    polygraph_model = WhiteboxModel(base_model, tokenizer, model_path=MODEL_ID, generation_parameters=shared_generation_params)
else:
  try:
        polygraph_model = BlackboxModel(
            model_path=MODEL_ID,
            openai_api_key=os.environ.get("OPENAI_API_KEY"),
            hf_api_token=os.environ.get("HF_TOKEN"),
            supports_logprobs= True,
            generation_parameters=shared_generation_params
        )
  except Exception as e:
        # Catching the exception
        print(f"WARNING: Failed to initialize BlackboxModel with logprobs. Error: {e}")
        print("Attempting graceful fallback to pure Black-box mode (supports_logprobs=False)...")

        # Fallback initialization without logprobs
        polygraph_model = BlackboxModel(
            model_path=MODEL_ID,
            openai_api_key=os.environ.get("OPENAI_API_KEY"),
            hf_api_token=os.environ.get("HF_TOKEN"),
            supports_logprobs=False,
            generation_parameters=shared_generation_params
        )

 Setting up lm_polygraph in WHITE-BOX mode...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [7]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
import types
import asyncio
from langchain_core.messages import SystemMessage, HumanMessage


print(f"📊 Setting up uqlm in {UQLM_MODE.upper()}-BOX mode...")

# Build the shared LangChain bridge
if PROVIDER == "openai":
    langchain_llm = ChatOpenAI(
        model=MODEL_ID,
        api_key=os.environ["OPENAI_API_KEY"],
        model_kwargs={"logprobs": True} if UQLM_MODE == "white" else {}
    )

elif PROVIDER == "huggingface":
    if UQLM_MODE == "white":
        print("   ↳ Bridging the LOCAL GPU model directly to LangChain...")
        pipe = pipeline(
            "text-generation",
            model=base_model,
            tokenizer=tokenizer,
            max_new_tokens=256,
            return_full_text=False,
            do_sample=True,
            temperature=0.7
        )

        local_hf_llm = HuggingFacePipeline(pipeline=pipe)

        langchain_llm = ChatHuggingFace(llm=local_hf_llm)

    else:
        print("   ↳ Using Remote HuggingFace API (Warning: May be unstable for async UQ)...")
        hf_endpoint = HuggingFaceEndpoint(
            repo_id=MODEL_ID,
            huggingfacehub_api_token=os.environ["HF_TOKEN"],
            task="text-generation",
            temperature=0.7,
            max_new_tokens=256,
            model_kwargs={
                "do_sample": True,
                "return_full_text": False
            }
        )
        langchain_llm = ChatHuggingFace(llm=hf_endpoint)



print("  Applying async patch")

gpu_lock = asyncio.Lock()

async def _agenerate_shim(self, messages, stop=None, run_manager=None, **kwargs):
    sanitized_messages = []
    system_buffer = ""

    for msg in messages:
        if isinstance(msg, SystemMessage):
            system_buffer += msg.content + "\n\n"
        elif isinstance(msg, HumanMessage):
            if system_buffer:
                msg.content = system_buffer + msg.content
                system_buffer = ""
            sanitized_messages.append(msg)
        else:
            sanitized_messages.append(msg)
    # --------------------------------------------------------------------

    async with gpu_lock:
        return await asyncio.to_thread(
            self._generate,
            sanitized_messages,
            stop=stop,
            run_manager=run_manager,
            **kwargs
        )

langchain_llm._agenerate = types.MethodType(_agenerate_shim, langchain_llm)



📊 Setting up uqlm in WHITE-BOX mode...
   ↳ Bridging the LOCAL GPU model directly to LangChain...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


  Applying async patch


### UQEngineContext

A centralized **Context Object** that manages the configuration and dependencies for the UQ framework.

* **State & Models:** Stores the execution modes (`"white"` or `"black"`) and holds the initialized model instances for both Polygraph and UQLM.
* **Auto-Validation:** Uses `__post_init__` to automatically enforce valid modes upon instantiation, ensuring a strict *fail-fast* architecture.

In [8]:
from dataclasses import dataclass
from typing import Any, Optional

@dataclass
class UQEngineContext:
    polygraph_mode: str  # "white" o "black"
    uqlm_mode: str       # "white" o "black"
    polygraph_model: Optional[Any] = None
    langchain_llm: Optional[Any] = None

    def __post_init__(self):
        if self.polygraph_mode not in ["white", "black"]:
            raise ValueError("'polygraph_mode' must be 'white' o 'black'.")
        if self.uqlm_mode not in ["white", "black"]:
            raise ValueError("'uqlm_mode' must be 'white' o 'black'.")

In [9]:
uq_engine = UQEngineContext(
    polygraph_mode="white",
    uqlm_mode="white",
    polygraph_model=polygraph_model,
    langchain_llm=langchain_llm
)

### UQ_REGISTRY & Access Hierarchy

A centralized dictionary that configures and routes all supported Uncertainty Quantification (UQ) techniques across different libraries.

* **Hierarchical Access (`MODE_LEVELS`):** Assigns numerical privilege levels (`black: 0`, `white: 1`) to enforce strict authorization, ensuring models meet the minimum required access to run a specific technique.


In [10]:
# --- Imports for lm_polygraph Execution ---
from lm_polygraph.estimators import *
from lm_polygraph import estimate_uncertainty

# ==========================================
# THE UQ REGISTRY (
# ==========================================

# Defining the hierarchy:
MODE_LEVELS = {
    "black": 0,
    "white": 1
}

UQ_REGISTRY = {
    # --- lm_polygraph techniques  ---
    "polygraph_attention": {
        "library": "lm_polygraph",
        "supported_granularity": ["sequence", "claim"],
        "min_required_mode": "white",
        "estimator_class": AttentionScore,
        "description": "Estimates uncertainty based on model’s attention weights."
    },
    "polygraph_max_token_prob": {
        "library": "lm_polygraph",
        "supported_granularity": ["token"],
        "min_required_mode": "black",
        "estimator_class": MaximumTokenProbability,
        "description": "Estimates token-level uncertainty by calculating log-probability."
    },

    # --- uqlm techniques ---
    "uqlm_exact_match": {
        "library": "uqlm",
        "supported_granularity": ["sequence"],
        "min_required_mode": "black",
        "wrapper_class": BlackBoxUQ,
        "description": "Black-box technique computing exact match consistency."
    },
    "uqlm_entailment": {
        "library": "uqlm",
        "supported_granularity": ["sequence", "claim"],
        "wrapper_class": {
            "sequence": BlackBoxUQ,
            "claim": LongTextUQ
        },
        "description": "Entailment Probability computes mean entailment via an NLI model."
    }
}

###  `show_help` (CLI Discovery Utility)

An internal helper function that enhances framework **discoverability** by rendering a clean, tabular summary of all registered UQ techniques directly in the terminal.

* **Dynamic Filtering:** Allows users to narrow down the available techniques by passing optional search parameters (`library`, `mode`, `granularity`).

In [11]:
def show_help(library: str = None, mode: str = None, granularity: str = None):
    """
    Internal CLI/Help function.
    Prints a formatted summary table of all registered UQ techniques.

    Optional parameters to filter the output:
    - library (str): e.g., "uqlm" or "lm_polygraph"
    - mode (str): e.g., "white" or "black" (filters by minimum requirement)
    - granularity (str): e.g., "token", "sequence", "claim"
    """
    print("\n🔍 UQ FRAMEWORK - AVAILABLE TECHNIQUES")

    # --- 1. Table Structure Definition ---
    # We use f-string alignment modifiers (e.g., <25 means "left-align, occupy 25 characters")
    header = f"{'TECHNIQUE':<26} | {'LIBRARY':<13} | {'MODE':<6} | {'GRANULARITY':<18} | {'DESCRIPTION'}"
    separator = "-" * 120

    print(separator)
    print(header)
    print(separator)

    count = 0
    for tech_name, info in UQ_REGISTRY.items():
        # --- 2. Safe Extraction (Fail-Safe) ---
        lib = info.get("library", "N/A")
        req_mode = info.get("min_required_mode", "N/A")
        gran = ", ".join(info.get("supported_granularity", []))
        desc = info.get("description", "No description provided.")

        # --- 3. Filtering Engine ---
        if library and lib.lower() != library.lower():
            continue

        if mode and req_mode.lower() != mode.lower():
            continue

        if granularity and granularity.lower() not in [g.lower() for g in info.get("supported_granularity", [])]:
            continue

        # --- 4. Visual Cleanup ---
        # Truncate the description if it's too long to keep the table clean in the terminal
        max_desc_len = 200
        if len(desc) > max_desc_len:
            desc = desc[:max_desc_len - 3] + "..."

        # Print the formatted row in columns
        row = f"{tech_name:<26} | {lib:<13} | {req_mode:<6} | {gran:<18} | {desc}"
        print(row)
        count += 1

    print(separator)
    print(f"Showing {count} techniques based on applied filters.\n")

In [9]:
show_help()


🔍 UQ FRAMEWORK - AVAILABLE TECHNIQUES
------------------------------------------------------------------------------------------------------------------------
TECHNIQUE                  | LIBRARY       | MODE   | GRANULARITY        | DESCRIPTION
------------------------------------------------------------------------------------------------------------------------
polygraph_attention        | lm_polygraph  | white  | sequence, claim    | Estimates uncertainty based on model’s attention weights.
polygraph_max_token_prob   | lm_polygraph  | black  | token              | Estimates token-level uncertainty by calculating log-probability.
uqlm_exact_match           | uqlm          | black  | sequence           | Black-box technique computing exact match consistency.
uqlm_entailment            | uqlm          | N/A    | sequence, claim    | Entailment Probability computes mean entailment via an NLI model.
---------------------------------------------------------------------------------------

### The Core Execution Engine (Dispatcher & Handlers)

A state-of-the-art routing architecture that separates validation logic from library-specific execution using the **Facade + Handlers** pattern.

* **`evaluate_uncertainty` (The Dispatcher):** The universal public interface to **compute the uncertainty score**. It performs *fail-fast* registry validation, enforces hierarchical access controls (preventing Privilege Escalation between `white` and `black` modes), and securely routes the request to the correct underlying library to generate the standardized UQ payload.
* **`_handle_polygraph_execution`:** The private sub-engine for `lm_polygraph`. It handles its specific synchronous API, complex token string decoding, and multi-step claim extraction pipelines.
* **`_handle_uqlm_execution`:** The private asynchronous sub-engine for `uqlm`. It dynamically resolves the correct wrapper class based on the requested granularity and parses the library's nested output dictionaries.

In [11]:
# ==========================================
#  AUXILIARY FUNCTIONS (Private logic)
# ==========================================
import os
import getpass
from lm_polygraph.stat_calculators import GreedyProbsCalculator, ClaimsExtractor
from lm_polygraph.utils.openai_chat import OpenAIChat

def _evaluate_claim_level_polygraph(prompt: str, estimator_class, model):
    """
    Handles the complex multi-step pipeline for Claim-level UQ in lm_polygraph.
    Requires OPENAI_API_KEY in the environment for the ClaimsExtractor.
    """
    print("   ↳ Initiating multi-step Claim-Level Pipeline...")

    if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"]:
        print("\n   ⚠️ Warning: Missing Open AI Key for ClaimsExtractor.")
        api_key = getpass.getpass("    Insert Open-AI key (sk-...): ")
        os.environ["OPENAI_API_KEY"] = api_key.strip()
        print("   Open AI key set!\n")
    # ---------------------------------------------

    stat = {}
    texts = [prompt]

    print("   ↳ Step 1: Generating text and probabilities (GreedyProbsCalculator)...")
    greedy_calc = GreedyProbsCalculator()
    stat.update(greedy_calc(stat, texts, model))

    print("   ↳ Step 2: Extracting atomic claims (ClaimsExtractor via GPT-4)...")
    extractor = ClaimsExtractor(OpenAIChat("gpt-4"))
    stat.update(extractor(stat, texts, model))

    print(f"   ↳ Step 3: Applying {estimator_class.__name__}...")
    estimator = estimator_class()
    uncertainties = estimator(stat)


    print("   ↳ Step 4: Formatting the output...")
    claims_list = stat["claims"][0]
    scores_list = uncertainties[0]

    claim_details = []
    for claim_obj, score in zip(claims_list, scores_list):
        claim_details.append({
            "claim_text": claim_obj.claim_text,
            "score": float(score)
        })

    result_payload = {
        "input_prompt": prompt,
        "generated_text": stat["greedy_texts"][0],
        "uncertainty_score": claim_details
    }

    return result_payload

# ==========================================
# 2. PRIVATE HANDLERS (The Sub-Engines)
# ==========================================

def _handle_polygraph_execution(prompt: str, tech_info: dict, granularity: str, polygraph_model, **kwargs):
    """Handles all execution and parsing specifically for lm_polygraph."""
    print(f"⏳ Routing to Wrapper A (lm_polygraph) -> {tech_info['estimator_class'].__name__}...")

    # --- CLAIM ---
    if granularity == "claim":
        result_payload = _evaluate_claim_level_polygraph(prompt, tech_info["estimator_class"](**kwargs), polygraph_model)
        result_payload["library"] = "lm_polygraph"
        result_payload["estimator_name"] = tech_info["estimator_class"].__name__
        result_payload["granularity"] = granularity
        return result_payload

    # --- TOKEN ---
    elif granularity == "token":
        estimator = tech_info["estimator_class"](**kwargs)
        output = estimate_uncertainty(polygraph_model, estimator, input_text=prompt)
        import numpy as np

        raw_score = output.uncertainty
        if isinstance(raw_score, list) and len(raw_score) > 0 and isinstance(raw_score[0], np.ndarray):
            clean_score_list = raw_score[0].tolist()
        elif isinstance(raw_score, np.ndarray):
            clean_score_list = raw_score.tolist()
        else:
            clean_score_list = list(raw_score)

        raw_tokens = output.generation_tokens
        if len(raw_tokens) == 1 and isinstance(raw_tokens[0], list):
            raw_tokens = raw_tokens[0]

        if len(raw_tokens) > 0 and isinstance(raw_tokens[0], int):
            token_strings = polygraph_model.tokenizer.convert_ids_to_tokens(raw_tokens)
        else:
            token_strings = raw_tokens

        token_details = []
        min_len = min(len(token_strings), len(clean_score_list))
        for i in range(min_len):
            clean_token = str(token_strings[i]).replace("Ġ", " ").replace(" ", " ")
            token_details.append({
                "token": clean_token,
                "score": float(clean_score_list[i])
            })

        return {
            "library": "lm_polygraph",
            "estimator_name": output.estimator,
            "granularity": granularity,
            "input_prompt": output.input_text,
            "generated_text": output.generation_text,
            "uncertainty_score": token_details,
        }

    # --- SEQUENCE ---
    else:
        estimator = tech_info["estimator_class"](**kwargs)
        output = estimate_uncertainty(polygraph_model, estimator, input_text=prompt)
        import numpy as np

        if isinstance(output.uncertainty, np.ndarray) or isinstance(output.uncertainty, list):
            final_score = float(output.uncertainty[0])
        else:
            final_score = float(output.uncertainty)

        return {
            "library": "lm_polygraph",
            "estimator_name": output.estimator,
            "granularity": granularity,
            "input_prompt": output.input_text,
            "generated_text": output.generation_text,
            "uncertainty_score": final_score
        }

In [12]:
async def _handle_uqlm_execution(prompt: str, technique_name: str, tech_info: dict, granularity: str, langchain_llm, **kwargs):
    """Handles all execution and parsing specifically for uqlm."""

    wrapper_map = tech_info["wrapper_class"]
    print(f"⏳ Routing to Wrapper B (uqlm) -> {uqlm_class.__name__} with scorer: '{technique_name}'...")

    uqlm_class = wrapper_map[granularity] if isinstance(wrapper_map, dict) else wrapper_map

    uqlm_wrapper = uqlm_class(
        llm=langchain_llm,
        scorers=[technique_name],
        **kwargs
    )

    uqlm_result = await uqlm_wrapper.generate_and_score(prompts=[prompt])
    res_dict = uqlm_result.to_dict()

    # --- CLAIM ---
    if granularity == "claim":
        print(f"res dict: {res_dict}")
        raw_claims_data = res_dict["data"]["claims_data"][0]
        claim_details = []
        for c in raw_claims_data:
            claim_details.append({
                "claim_text": c['claim'],
                "score": c[technique_name]
            })

        return {
            "library": "uqlm",
            "estimator_name": technique_name,
            "granularity": granularity,
            "input_prompt": prompt,
            "generated_text": res_dict["data"]["responses"],
            "uncertainty_score": claim_details
        }

    # --- TOKEN ---
    elif granularity == "token":
        return "UQLM do not support token level granularuty"

    # --- SEQUENCE ---
    else:
        return {
            "library": "uqlm",
            "estimator_name": technique_name,
            "granularity": granularity,
            "input_prompt": prompt,
            "generated_text": res_dict["data"]["responses"],
            "uncertainty_score": res_dict["data"][technique_name][0]
        }



In [ ]:
async def evaluate_uncertainty(prompt: str, technique_name: str, library: str, granularity: str,
                         uq_context: UQEngineContext, **kwargs):
    """
    Universal interface for UQ evaluation. Routes to simple functions or
    complex pipelines based on the requested granularity, standardizing the output.
    """
    print(f"\n🧠 Processing Request: Library='{library}' | Technique='{technique_name}' | Granularity='{granularity}'")

    registry_key = f"{library}_{technique_name}"

    # --- Step A: Registry Validation ---
    if registry_key not in UQ_REGISTRY:
        raise ValueError(f"Technique combination '{registry_key}' is not in the UQ_REGISTRY. Please add it first.")

    tech_info = UQ_REGISTRY[registry_key]

    if granularity not in tech_info["supported_granularity"]:
        raise ValueError(
            f" Granularity Mismatch: '{technique_name}' in {library} only supports {tech_info['supported_granularity']}. "
            f"You requested '{granularity}'."
        )

    current_mode = getattr(uq_context, f"{library}_mode")
    min_required_mode = tech_info["min_required_mode"]

    # Translate mode strings into their corresponding numeric levels
    current_level = MODE_LEVELS.get(current_mode, -1)
    required_level = MODE_LEVELS.get(min_required_mode, 99)

    # If the current level is lower than the required one, block execution!
    if current_level < required_level:
        raise ValueError(
            f"Privilege Escalation Error: The technique '{registry_key}' requires "
            f"'{min_required_mode.upper()}' level access, but the library '{library}' "
            f"is initialized at a lower level ('{current_mode.upper()}')."
        )
    # ----------------------------------------------

    print(f"Validation Passed. Library to use: {tech_info['library'].upper()} (Mode: {current_mode})")

    # --- Step B: Execution Routing ---
    if tech_info["library"] == "lm_polygraph":
        if uq_context.polygraph_model is None:
            raise ValueError("'polygraph_model' is required by this technique but was not found in the UQEngineContext.")

        result_payload = _handle_polygraph_execution(prompt, tech_info, granularity, uq_context.polygraph_model, **kwargs)

    elif tech_info["library"] == "uqlm":
        if uq_context.langchain_llm is None:
              raise ValueError(" 'langchain_llm' bridge is required by this technique but was not found in the UQEngineContext.")

        result_payload = await _handle_uqlm_execution(prompt, technique_name, tech_info, granularity, uq_context.langchain_llm, **kwargs)

    print(f"🎯 {granularity.capitalize()}-level execution complete!")
    return result_payload

In [23]:
test_result = await evaluate_uncertainty(
    prompt="What are the early signs of Parkinson's disease?",
    library="uqlm",
    technique_name="entailment",
    granularity="claim",
    uq_context=uq_engine
)
print(test_result)



🧠 Processing Request: Library='uqlm' | Technique='entailment' | Granularity='claim'
Validation Passed. Library to use: UQLM (Mode: white)
⏳ Routing to Wrapper B (uqlm) -> LongTextUQ with scorer: 'entailment'...


Output()

res dict: {'data': {'responses': ['Parkinson\'s disease is a complex neurological disorder, and it\'s important to remember that early signs can be subtle and vary from person to person. If you\'re concerned about potential symptoms, it\'s crucial to consult a healthcare professional for proper diagnosis and guidance. \n\nHere are some early signs of Parkinson\'s disease:\n\n**Motor Symptoms:**\n\n* **Tremor:** This is often the first noticeable sign, usually a rhythmic shaking or trembling, often affecting one hand.\n* **Rigid Muscles:** Stiffness and difficulty with movement, leading to slow, deliberate movements and stiffness in the limbs.\n* **Slow Movement (Bradykinesia):**  Slowness of movement, making everyday tasks like walking, talking, or buttoning a shirt more challenging.\n* **Postural Instability:** Difficulty maintaining balance, leading to an increased risk of falls.\n* **Postural Changes:**  A stooped posture, hunching forward.\n* **Reduced Flexibility and Flexibility:*

# Text-Only

## Black Box Techniques

## White Box Techniques

# Multimodal


#Normalization strategies

# Benchmarking